In [1]:
#!/usr/bin/env python3
import os
import torch
from equiv_dens.training.parse_command_line_arguments import parse_command_line_arguments
from equiv_dens.training.errors import ErrorDict
from equiv_dens.data.density_dataset import AtomsDensityData
from equiv_dens.data.hamiltonian_dataset import seeded_random_split
from equiv_dens.utils.grids import cubical_grid, cubical_sampling,\
    dftpy_grid, CubicalGrid, spherical_grid, spherical_radial_sampling
from equiv_dens.training.model_loader import load_model
import equiv_dens.utils.base as utils

import numpy as np
from functools import partial

%load_ext autoreload
%autoreload 2

Use "numpy" for Fourier Transform


/home/mihail/anaconda3/envs/equiv_dens/lib/python3.7/site-packages/pyscf/lib/misc.py:47: H5pyDeprecationWarning: Using default_file_mode other than 'r' is deprecated. Pass the mode to h5py.File() instead.
  h5py.get_config().default_file_mode = 'a'


In [141]:
args, hyperparam_args = parse_command_line_arguments(arg_file='ethanol_dens_001_mae_test.txt')
#args, hyperparam_args = parse_command_line_arguments(arg_file='ethanethiol_dft_pyscf_dens11_spherical_new_3.txt')

print('type dtype', type(args.dtype))
args.fix_arguments = True
print('args np dir', args.np_dataset)
# no restart directory specified
directory = args.restart  # load directory name
# load latest checkpoint
checkpoint_path = os.path.join(directory, 'checkpoints')  # checkpoint directory
checkpoint = torch.load(os.path.join(
    checkpoint_path, 'latest_checkpoint.pth'), map_location='cpu')
latest_checkpoint = checkpoint['step']
model_code = checkpoint['ID']  # load ID
step = checkpoint['step']
for arg in vars(checkpoint['args']):
    if args.fix_arguments:
        if arg in hyperparam_args:
            print('loading hyperparam arg', arg)
            setattr(args, arg, getattr(checkpoint['args'], arg))
    else:
        print('loading all arg', arg)
        setattr(args, arg, getattr(checkpoint['args'], arg))
restore = True

args.best_model_path = 'best_' + model_code + '.pth'
print('best_model_path', args.best_model_path)

print('model code:', model_code)
# determine whether GPU is used for training
print('args use gpu', args.use_gpu)
args.use_gpu = args.use_gpu and torch.cuda.is_available()
np_dataset = np.load(args.np_dataset, allow_pickle=True).item()
min_pos = np.min(np_dataset['positions'])
print(min_pos)
max_pos = np.max(np_dataset['positions'])
print(max_pos)
# load dataset(s)
print("loading density from" + str(args.dens_dataset) + "...")
print("loading atoms from" + args.np_dataset + "...")
args.verbose = 0
args.use_gpu = False
args.cube_grid = False 
args.radii_adjust = True 
if args.cube_grid:
    args.cube_origin = min_pos - 1
    args.cube_extent = max_pos-min_pos + 2
    args.cube_size = 50
    args.radii_adjust = False
    grid_origin = args.cube_origin
    grid_extent = np.array([args.cube_extent] * 3)
    grid_fn = partial(cubical_grid, nx=args.cube_size, ny=args.cube_size, nz=args.cube_size,
                      extent=grid_extent,
                      origin=np.array([grid_origin] * 3))
    sampling_fn = cubical_sampling
else:
    grid_fn = partial(spherical_grid, level=6)
    sampling_fn = partial(spherical_radial_sampling, rotate=False)
    grid_origin = 0
    grid_extent = None
    
args.integral_constraint = True
args.restart = None

dataset = AtomsDensityData(np_path=args.np_dataset, density_path=args.dens_dataset,
                           orbitals_path=args.orbitals_file,
                           density_n_samp=10000000000,
                           required_properties=['density'],
                           center_positions=False,
                           radial_coeffs_file=args.radial_coeffs_file,
                           dtype=args.dtype,
                           grid_fn=grid_fn,
                           sampling_fn=sampling_fn,
                           grid_extent=grid_extent,
                           grid_origin=grid_origin,
                           verbose=args.verbose,
                           radii_adjust=args.radii_adjust)

type dtype <class 'torch.dtype'>
args np dir datasets/ethanol_dft_train.npy
loading hyperparam arg activation
loading hyperparam arg order
loading hyperparam arg mixing_order
loading hyperparam arg order_en
loading hyperparam arg mixing_order_en
loading hyperparam arg num_features
loading hyperparam arg num_basis_functions
loading hyperparam arg num_radial_components
loading hyperparam arg num_energy_features
loading hyperparam arg num_modules
loading hyperparam arg num_residual_pre_x
loading hyperparam arg num_residual_post_x
loading hyperparam arg num_residual_pre_vi
loading hyperparam arg num_residual_pre_vj
loading hyperparam arg num_residual_post_v
loading hyperparam arg num_residual_output
loading hyperparam arg num_energy_output
loading hyperparam arg basis_functions
loading hyperparam arg cutoff
loading hyperparam arg orthonormal_basis
loading hyperparam arg expansion_constraint
loading hyperparam arg integral_constraint
loading hyperparam arg integral_scale
loading hyperparam 

In [142]:
model = load_model(args, dataset)

cg_matrix shape torch.Size([121, 121, 121])
args energy_unit_in kcal
args energy_unit_out kcal
conversions in <function kcal_to_kcal at 0x7fbd274618c8>
conversions out <function kcal_to_kcal at 0x7fbd274618c8>
self order [1, 3, 5]
self order [1, 3, 5]
self mixing_order [1, 3, 5]
self mixing_order [1, 3, 5]
creating embedding
init_coeffs None
orbital basis {6: [(6, 1, 0), (6, 1, 0), (6, 1, 0), (6, 1, 0), (6, 1, 0), (6, 1, 0), (6, 1, 0), (6, 1, 0), (6, 1, 0), (6, 1, 0), (6, 1, 0), (6, 1, 1), (6, 1, 1), (6, 1, 1), (6, 1, 1), (6, 1, 1), (6, 1, 1), (6, 1, 1), (6, 1, 1), (6, 1, 2), (6, 1, 2), (6, 1, 2), (6, 1, 2), (6, 1, 2), (6, 1, 2), (6, 1, 3), (6, 1, 3), (6, 1, 3), (6, 1, 3), (6, 1, 4), (6, 1, 4), (6, 1, 4), (6, 1, 5), (6, 1, 5)], 8: [(8, 1, 0), (8, 1, 0), (8, 1, 0), (8, 1, 0), (8, 1, 0), (8, 1, 0), (8, 1, 0), (8, 1, 0), (8, 1, 0), (8, 1, 0), (8, 1, 0), (8, 1, 1), (8, 1, 1), (8, 1, 1), (8, 1, 1), (8, 1, 1), (8, 1, 1), (8, 1, 1), (8, 1, 1), (8, 1, 2), (8, 1, 2), (8, 1, 2), (8, 1, 2), (8, 1

In [143]:
train_indices = checkpoint['data_split_indices']['train']

In [151]:
torch.set_printoptions(precision=4)
sample = dataset.get_properties([train_indices[124]])
results = model(sample)
print('true density integral', torch.sum(sample['density'] * results['coord_weights']))
print('results density integral', torch.sum(results['density'] * results['coord_weights']))
print('density error', torch.sum(torch.abs(results['density'] - sample['density']) * sample['coord_weights'])/torch.sum(sample['atom_numbers']))
scaled_density = results['density']/torch.sum(results['density'] * results['coord_weights']) * torch.sum(results['atom_numbers'])
print('scaled density integral', torch.sum(scaled_density * results['coord_weights']))
print('density error with scaling', torch.sum(torch.abs(scaled_density - sample['density']) * sample['coord_weights'])/torch.sum(sample['atom_numbers']))

i 0  L0_i 0
L0_dens integral tensor([2.9332], grad_fn=<SumBackward1>)
l0 dens shape torch.Size([1, 402192])
atoms density shape torch.Size([1, 402192])
i 1  L0_i 1
L0_dens integral tensor([2.9989], grad_fn=<SumBackward1>)
l0 dens shape torch.Size([1, 402192])
atoms density shape torch.Size([1, 402192])
i 2  L0_i 2
L0_dens integral tensor([-22.6688], grad_fn=<SumBackward1>)
l0 dens shape torch.Size([1, 402192])
atoms density shape torch.Size([1, 402192])
i 3  L0_i 3
L0_dens integral tensor([7.0425], grad_fn=<SumBackward1>)
l0 dens shape torch.Size([1, 402192])
atoms density shape torch.Size([1, 402192])
i 4  L0_i 4
L0_dens integral tensor([7.0883], grad_fn=<SumBackward1>)
l0 dens shape torch.Size([1, 402192])
atoms density shape torch.Size([1, 402192])
i 5  L0_i 5
L0_dens integral tensor([7.1304], grad_fn=<SumBackward1>)
l0 dens shape torch.Size([1, 402192])
atoms density shape torch.Size([1, 402192])
i 6  L0_i 6
L0_dens integral tensor([7.1572], grad_fn=<SumBackward1>)
l0 dens shape to

In [28]:
print(args.integral_constraint)

True


In [29]:
print(model.property_models['density'].integral_constraint)

True


In [152]:
torch.set_printoptions(precision=10)
width = torch.logspace(-1, 5, 100)
width = width.unsqueeze(0).unsqueeze(0)
print('width shape', width.shape)
scale_calc = (width**(3 / 2)) / (np.pi**(3 / 2)) * utils.to_angstrom**3
print('scale shape', scale_calc.shape)

# center = torch.mean(sample['coords'], dim=1)
center = sample['positions'][0, [0]]
print('center', center)
print('center shape', center.shape)
print('sample coords shape', sample['coords'].shape)
r = torch.sum((center.unsqueeze(0) - sample['coords'])**2, dim=-1)
r = r.unsqueeze(-1)
print('r shape', r.shape)
rbf = scale_calc * torch.exp(-width * r)
print('rbf shape', rbf.shape)

rbf_int = torch.sum(rbf * sample['coord_weights'].unsqueeze(-1), dim=1)
print('rbf_int shape', rbf_int.shape)
print('int_vs_width', torch.cat([width.squeeze(0),rbf_int], dim=0).T)

width shape torch.Size([1, 1, 100])
scale shape torch.Size([1, 1, 100])
center tensor([[ 0.1238970608,  0.4036535919, -0.3058898449]])
center shape torch.Size([1, 3])
sample coords shape torch.Size([1, 402192, 3])
r shape torch.Size([1, 402192, 1])
rbf shape torch.Size([1, 402192, 100])
rbf_int shape torch.Size([1, 100])
int_vs_width tensor([[1.0000000149e-01, 1.0000026226e+00],
        [1.1497569829e-01, 9.9999886751e-01],
        [1.3219411671e-01, 9.9999952316e-01],
        [1.5199111402e-01, 9.9999994040e-01],
        [1.7475284636e-01, 9.9999982119e-01],
        [2.0092329383e-01, 9.9999994040e-01],
        [2.3101297021e-01, 9.9999982119e-01],
        [2.6560878754e-01, 1.0000001192e+00],
        [3.0538555980e-01, 9.9999994040e-01],
        [3.5111916065e-01, 1.0000000000e+00],
        [4.0370172262e-01, 9.9999994040e-01],
        [4.6415889263e-01, 1.0000000000e+00],
        [5.3366994858e-01, 1.0000001192e+00],
        [6.1359071732e-01, 1.0000001192e+00],
        [7.054802179

In [153]:
torch.set_printoptions(precision=10)
width = torch.logspace(-1, 5, 100)
width = width.unsqueeze(0).unsqueeze(0)
print('width shape', width.shape)
scale_calc = (width**(3 / 2)) / (np.pi**(3 / 2)) * utils.to_angstrom**3
print('scale shape', scale_calc.shape)

# center = torch.mean(sample['coords'], dim=1)
center = sample['positions'][0, [0]]
print('center', center)
print('center shape', center.shape)
print('sample coords shape', sample['coords'].shape)
r = torch.sum((center.unsqueeze(0) - sample['coords'])**2, dim=-1)
r = r.unsqueeze(-1)
print('r shape', r.shape)
rbf = scale_calc * torch.exp(-width * r)
print('rbf shape', rbf.shape)

rbf_int = torch.sum(rbf * sample['coord_weights'].unsqueeze(-1), dim=1)
print('rbf_int shape', rbf_int.shape)
print('int_vs_width', torch.cat([width.squeeze(0),rbf_int], dim=0).T)


width shape torch.Size([1, 1, 100])
scale shape torch.Size([1, 1, 100])
center tensor([[ 0.1238970608,  0.4036535919, -0.3058898449]])
center shape torch.Size([1, 3])
sample coords shape torch.Size([1, 402192, 3])
r shape torch.Size([1, 402192, 1])
rbf shape torch.Size([1, 402192, 100])
rbf_int shape torch.Size([1, 100])
int_vs_width tensor([[1.0000000149e-01, 1.0000026226e+00],
        [1.1497569829e-01, 9.9999886751e-01],
        [1.3219411671e-01, 9.9999952316e-01],
        [1.5199111402e-01, 9.9999994040e-01],
        [1.7475284636e-01, 9.9999982119e-01],
        [2.0092329383e-01, 9.9999994040e-01],
        [2.3101297021e-01, 9.9999982119e-01],
        [2.6560878754e-01, 1.0000001192e+00],
        [3.0538555980e-01, 9.9999994040e-01],
        [3.5111916065e-01, 1.0000000000e+00],
        [4.0370172262e-01, 9.9999994040e-01],
        [4.6415889263e-01, 1.0000000000e+00],
        [5.3366994858e-01, 1.0000001192e+00],
        [6.1359071732e-01, 1.0000001192e+00],
        [7.054802179

In [107]:
print('init widths', model.property_models['density'].init_width_1_0)

init widths tensor([[[[9.5302495956, 1.9174506664, 0.6842404604, 0.2841325700, 0.1179867536]]]])


In [157]:
width = torch.cat([model.property_models['density'].init_width_8_0, model.property_models['density'].init_width_6_0, model.property_models['density'].init_width_1_0], dim=-1).squeeze(0)
width = torch.clamp(width, 1e-1, 1e+5)
print('width shape', width.shape)
scale_calc = (width**(3 / 2)) / (np.pi**(3 / 2)) * utils.to_angstrom**3
print('scale shape', scale_calc.shape)

# center = torch.mean(sample['coords'], dim=1)
center = sample['positions'][0, [0]]
print('center', center)
print('center shape', center.shape)
print('sample coords shape', sample['coords'].shape)
r = torch.sum((center.unsqueeze(0) - sample['coords'])**2, dim=-1)
r = r.unsqueeze(-1)
print('r shape', r.shape)
rbf = scale_calc * torch.exp(-width * r)
print('rbf shape', rbf.shape)

rbf_int = torch.sum(rbf * sample['coord_weights'].unsqueeze(-1), dim=1)
print('rbf_int shape', rbf_int.shape)
print('int_vs_width', torch.cat([width.squeeze(0),rbf_int], dim=0).T)
print('rbf_int', rbf_int)
print('mean_factor', torch.mean(rbf_int))
print('median_factor', torch.median(rbf_int))

width shape torch.Size([1, 1, 27])
scale shape torch.Size([1, 1, 27])
center tensor([[ 0.1238970608,  0.4036535919, -0.3058898449]])
center shape torch.Size([1, 3])
sample coords shape torch.Size([1, 402192, 3])
r shape torch.Size([1, 402192, 1])
rbf shape torch.Size([1, 402192, 27])
rbf_int shape torch.Size([1, 27])
int_vs_width tensor([[1.5178666992e+03, 1.0000000000e+00],
        [4.8967953491e+02, 9.9999994040e-01],
        [1.7672119141e+02, 1.0000000000e+00],
        [6.3792232513e+01, 1.0000000000e+00],
        [2.5366498947e+01, 1.0000000000e+00],
        [9.9135494232e+00, 1.0000000000e+00],
        [4.4645304680e+00, 1.0000000000e+00],
        [1.8017743826e+00, 1.0000001192e+00],
        [8.0789709091e-01, 1.0000001192e+00],
        [3.3864328265e-01, 1.0000000000e+00],
        [1.4194786549e-01, 9.9999976158e-01],
        [1.1139868164e+03, 1.0000000000e+00],
        [3.6916235352e+02, 1.0000000000e+00],
        [1.2179275513e+02, 1.0000000000e+00],
        [4.8127113342e+0

In [113]:
factor = torch.median(rbf_int)

In [158]:
print(1/factor)

tensor(0.1481847465)


In [149]:
print(1/utils.to_bohr**3)

0.1481847433778473


In [150]:
print(utils.to_angstrom**3)

0.14818474347690475
